# Figures for Wei, Mitchell & Maunsell (2023)

Reproduces the human psychophysics figures from the package. Two threshold sources are used:

- **staircase thresholds** computed by `dataIO.allSubjectDict` from the raw `.mat` files (needs `DATA_DIR`)
- **psignifit thresholds** estimated in MATLAB and shipped with the package (`dataIO.loadPsignifit`), which is what the published Fig 2 and Sup Fig 1 use

Cells that need the raw data are marked; everything else runs from the packaged psignifit values.

In [ ]:
from pathlib import Path
import numpy as np
from IDPsych import calc, cohorts, dataIO, vis

DATA_DIR = Path('../data')     # <- folder with one sub-folder per subject (raw data only)
OUT_DIR = Path('../figures')
OUT_DIR.mkdir(exist_ok=True)

sj2 = cohorts.subjects('100ms')   # 100 ms dot life, 2.5 dots/deg^2
sj4 = cohorts.subjects('33ms')    # 33 ms dot life, 5 dots/deg^2 (main dataset)
sj6 = cohorts.subjects('control') # increment-only control sessions

## Published figures (psignifit thresholds)

In [ ]:
psig4 = dataIO.loadPsignifit('33ms')
psig2 = dataIO.loadPsignifit('100ms')
calc.subjectSummary(psig4).round(1)

### Fig 2 — 33 ms cohort

In [ ]:
vis.thresholdScatter(psig4, saveTo=OUT_DIR / 'IDPsych_Fig2_psignifit.pdf');

### Sup Fig 1 — 100 ms cohort (subject 201 as open symbol)

In [ ]:
openSym = {sj: dict(mfc='w') for sj in cohorts.OPEN_SYMBOL['100ms']}
vis.thresholdScatter(psig2, style=openSym, saveTo=OUT_DIR / 'IDPsych_supFig1_psignifit.pdf');

### Learning effect

Do thresholds keep dropping across the 15 sessions, or have they leveled off? Exponential fit per subject and to the across-subject average.

In [ ]:
vis.learningAverage(psig4, saveTo=OUT_DIR / 'IDPsych_AveragePsychLearn_psig.pdf');

In [ ]:
calc.learningTable(psig4)

In [ ]:
vis.learningBySubject(psig4);

## Staircase thresholds (requires raw data)

Same figures using the last-trial staircase threshold from the `.mat` files, plus figures that need trial-level data.

In [ ]:
allData_2 = dataIO.allSubjectDict(DATA_DIR, sj2)
allData_4 = dataIO.allSubjectDict(DATA_DIR, sj4)
allData_6 = dataIO.allSubjectDict(DATA_DIR, sj6)
calc.subjectSummary(allData_4).round(1)

In [ ]:
vis.thresholdScatter(allData_4, saveTo=OUT_DIR / 'IDPsych_Fig2_staircase.pdf');

In [ ]:
vis.thresholdScatter(allData_2, style=openSym, saveTo=OUT_DIR / 'IDPsych_supFig1_staircase.pdf');

Early-draft colored version: subjects 403 and 408 (later used for the control) in red, the rest in blue.

In [ ]:
vis.thresholdScatterColored(allData_4, {vis.BLUE: ['404', '405', '406'], vis.RED: ['403', '408']});

### Control comparison

Increment thresholds from the control sessions (603, 608; baseline = 50% − subject's Dec threshold) against the matched subjects' decrement thresholds from their first 30 sessions (403, 408).

In [ ]:
def firstDec(sjData, n=30):
    tm = np.asarray(sjData['taskMode'][:n]); thr = np.asarray(sjData['thresholdPC'][:n])
    return thr[tm == 0]

control = {
    '603': dataIO.fromThresholds(allData_6['603']['thresholdPC'], firstDec(allData_4['403'])),
    '608': dataIO.fromThresholds(allData_6['608']['thresholdPC'], firstDec(allData_4['408'])),
}
red = {sj: dict(mfc=vis.RED, mec=vis.RED, color=vis.RED) for sj in control}
vis.thresholdScatter(control, style=red, xlabel='Increment Threshold (%)', ylabel='Decrement\nThreshold\n(%)',
                     saveTo=OUT_DIR / 'IDPsych_threshold_scatter_control.pdf');

### Learning effect, staircase thresholds

In [ ]:
vis.learningNormalized(allData_4);

In [ ]:
calc.learningTable(allData_4)

### Session diagnostics

In [ ]:
vis.hitRateHist(allData_4);
vis.rBias(allData_4);